# Experiment 1: does the model know how good its algorithms are?

Analysis of layer-wise linear probes on Qwen3-8B residual-stream activations recorded while LLaMEA designs MA-BBOB optimisers.
Produced by `scripts/03_train_probes.py`; set `TAG` to the campaign (`pilot`, `main`, ...).

**How to read the scores.** Every probe curve comes with a shuffled-label control on the same folds (gray, dashed).
The best layer is chosen after seeing the scores, which biases it upwards, so judge the gap to the control curve across layers and not the single maximum.
Folds are grouped by LLaMEA run.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MI_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(MI_ROOT))
from common.paths import data_dir, results_dir
from common.plotting import INK, NEUTRAL, POSITION_COLORS, POSITION_LABELS, setup

setup()
TAG = "pilot"
EXP = "exp01_probing"
res = results_dir(EXP, TAG)
meta = pd.read_csv(data_dir(EXP, "processed", TAG) / "meta.csv", dtype={"record_id": str})
scores = pd.read_csv(res / "probe_scores.csv")
baselines = pd.read_csv(res / "baselines.csv")
print(f"{len(meta)} labelled queries from {meta['run'].nunique()} runs")

## 1. Dataset: labels per operator

In [ ]:
summary = meta.groupby("operator").agg(
    n=("record_id", "size"),
    failed=("child_failed", "mean"),
    over_budget=("over_budget", "mean"),
    improved=("improved", "mean"),
    child_fitness=("child_fitness", "median"),
)
summary.round(3)

## 2. Layer-wise probe performance

One panel per target. Coloured lines: probe at each recording position; gray dashed: shuffled-label control; gray dotted: activation-free baselines.

In [ ]:
def metric_of(kind):
    return ("auroc", "AUROC", 0.5) if kind == "binary" else ("spearman", "Spearman ρ", 0.0)

targets = list(dict.fromkeys(scores["target"]))
fig, axes = plt.subplots(1, len(targets), figsize=(4.2 * len(targets), 3.6), squeeze=False)
for ax, target in zip(axes[0], targets):
    t = scores[scores["target"] == target]
    metric, label, chance = metric_of(t["kind"].iloc[0])
    for pos, g in t.groupby("position", sort=False):
        g = g.sort_values("layer")
        c = POSITION_COLORS[pos]
        ax.plot(g["layer"], g[metric], color=c, label=POSITION_LABELS[pos])
        ax.fill_between(g["layer"], g[metric] - g[f"{metric}_std"], g[metric] + g[f"{metric}_std"], color=c, alpha=0.12, linewidth=0)
        ax.text(g["layer"].iloc[-1] + 0.5, g[metric].iloc[-1], POSITION_LABELS[pos], color=INK, fontsize=8, va="center")
    ctrl = t.groupby("layer")[f"control_{metric}"].mean()
    ax.plot(ctrl.index, ctrl.values, color=NEUTRAL, linestyle="--", linewidth=1.5, label="shuffled control")
    b = baselines[baselines["target"] == target]
    for _, row in b.iterrows():
        if pd.notna(row.get(metric)):
            ax.axhline(row[metric], color=NEUTRAL, linestyle=":", linewidth=1.2)
            ax.text(0, row[metric], f" {row['baseline']}", color=INK, fontsize=8, va="bottom")
    ax.axhline(chance, color=NEUTRAL, linewidth=0.8)
    ax.set_title(f"{target} (n={t['n'].iloc[0]})", color=INK, fontsize=10, loc="left")
    ax.set_xlabel("layer")
    ax.set_ylabel(label)
axes[0][0].legend(loc="lower left", fontsize=8)
fig.tight_layout()
fig.savefig(res / "layer_curves.png", bbox_inches="tight")

In [ ]:
# Table view: best layer per target/position next to control and baselines.
idx = scores.assign(key=scores["auroc"].fillna(scores["spearman"])).groupby(["target", "position"])["key"].idxmax()
cols = ["target", "position", "layer", "n", "auroc", "control_auroc", "spearman", "control_spearman"]
scores.loc[idx, [c for c in cols if c in scores]].round(3)

In [ ]:
baselines.pivot_table(index="target", columns="baseline", values=["auroc", "spearman"]).round(3)

## 3. Pre-screening: could the probe save evaluation budget?

Using out-of-fold failure predictions (best layer; optimistic, see above): skip the evaluation of the fraction *k* of children with the highest predicted failure score.
The question is how much budget is saved, and how many working (and top-decile) algorithms are lost along the way.

In [ ]:
def prescreen_curve(oof, label):
    order = np.argsort(-oof["pred"].to_numpy())
    bad = oof[label].to_numpy()[order]
    fit = meta.set_index("record_id").loc[oof["record_id"], "child_fitness"].to_numpy()[order]
    top = fit >= np.nanquantile(fit, 0.9)
    ks = np.linspace(0, 1, 51)
    rows = []
    for k in ks:
        m = int(round(k * len(order)))
        rows.append({
            "skipped": k,
            "failures_caught": bad[:m].sum() / max(bad.sum(), 1),
            "working_lost": (1 - bad[:m]).sum() / max((1 - bad).sum(), 1),
            "top_decile_lost": top[:m].sum() / max(top.sum(), 1),
        })
    return pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, pos in zip(axes, ["last", "gen_mean"]):
    path = res / f"oof_child_failed_{pos}.csv"
    if not path.exists():
        ax.set_visible(False)
        continue
    oof = pd.read_csv(path, dtype={"record_id": str})
    cur = prescreen_curve(oof, "child_failed")
    for col, c in [("failures_caught", POSITION_COLORS["last"]), ("working_lost", POSITION_COLORS["prompt_mean"]), ("top_decile_lost", POSITION_COLORS["gen_mean"])]:
        ax.plot(cur["skipped"], cur[col], color=c, label=col.replace("_", " "))
    ax.plot([0, 1], [0, 1], color=NEUTRAL, linestyle="--", linewidth=1, label="random skipping")
    ax.set_title(f"{POSITION_LABELS[pos]} (layer {oof['layer'].iloc[0]})", color=INK, fontsize=10, loc="left")
    ax.set_xlabel("fraction of evaluations skipped")
    ax.set_ylabel("fraction")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(res / "prescreening.png", bbox_inches="tight")

## 4. Geometry: PCA of the last-token state at the best parent-fitness layer

In [ ]:
acts = np.load(data_dir(EXP, "processed", TAG) / "activations.npz")
t = scores[(scores["target"] == "parent_fitness") & (scores["position"] == "last")]
if len(t):
    layer = int(t.loc[t["spearman"].idxmax(), "layer"])
    li = list(acts["layers"]).index(layer)
    mask = meta["parent_fitness"].notna().to_numpy()
    X = acts["last"][mask, li].astype(np.float32)
    X = (X - X.mean(0)) / (X.std(0) + 1e-6)
    U, S, _ = np.linalg.svd(X, full_matrices=False)
    pc = U[:, :2] * S[:2]
    fig, ax = plt.subplots(figsize=(4.8, 4))
    sc = ax.scatter(pc[:, 0], pc[:, 1], c=meta.loc[mask, "parent_fitness"], cmap="Blues", s=14, edgecolors="white", linewidths=0.4)
    fig.colorbar(sc, label="parent fitness (AOCC)")
    ax.set_title(f"layer {layer}, last prompt token", color=INK, fontsize=10, loc="left")
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
    fig.savefig(res / "pca_parent_fitness.png", bbox_inches="tight")

## Notes

- *Parent fitness* is written in the prompt as a number, so it being decodable mainly shows the model tracks it at the decision point; the interesting part is **where** (which layers) and whether it survives into `gen_mean`.
- *child_failed / over_budget / improved* depend on sampling, so their ceiling is below 1. Compare against the `uncertainty` and `code_bow` baselines: activations only matter if they beat the code text.
- Next: fix the layer on the pilot, then report unbiased scores on `main` (or nested selection).